[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/06_linear_regression/03_normal_equation_and_gd_tips.ipynb)

# Bài 03 — Gradient Descent Tips và Normal Equation

**Nội dung**
1. Mẹo dùng Gradient Descent: chọn learning rate, kiểm tra hội tụ *(từ AIVN, bổ sung thực nghiệm)*
2. Normal Equation — nghiệm đóng, *(bổ sung)* chứng minh, ý nghĩa hình học
3. So sánh Gradient Descent và Normal Equation *(đã đính chính độ phức tạp)*
4. *(bổ sung)* Khi $X^\top X$ không khả nghịch

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/06_linear_regression

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml_course.visualization import set_style

set_style()

## 1. Gradient Descent Tips

*Nhắc lại:* công thức cập nhật $\theta$ ở vòng lặp thứ $t$:

$$\theta_{t+1} := \theta_t - \alpha\,\nabla_{\theta} J(\theta_t)$$

Việc chọn $\alpha$ (learning rate) rất quan trọng. Bạn có thể thử thay đổi trực quan [tại đây](https://developers.google.com/machine-learning/crash-course/fitter/graph).

**Learning rate quá lớn** — GD không hội tụ được về minimum:

<img src="images/image-3.gif" style="width:40%;height:40%;">

**Learning rate quá nhỏ** — hội tụ được nhưng mất tới 81 vòng lặp; với bài toán nhiều cực tiểu địa phương, LR quá nhỏ còn có thể kẹt ở cực tiểu xấu:

<img src="images/image-5.png" style="width:40%;height:40%;">

**Learning rate vừa** — ví dụ trên với LR = 1.0 chỉ mất 6 vòng lặp:

<img src="images/image-4.png" style="width:40%;height:40%;">

**Learning rate tối ưu** — rất khó tìm chính xác, nhưng tìm được giá trị *gần* tối ưu sẽ giúp hội tụ nhanh hơn nhiều:

<img src="images/image-6.png" style="width:40%;height:40%;">

### Thực nghiệm: quét learning rate theo thang log *(bổ sung)*

Cách làm thực tế: thử $\alpha \in \{0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, ...\}$ (mỗi bước ×3), vẽ $J$ theo vòng lặp, chọn giá trị **lớn nhất** mà $J$ vẫn giảm đều.

In [ ]:
df = pd.read_csv("../../data/sample/house_prices.csv")
features = ["area_m2", "bedrooms", "bathrooms", "age_years", "distance_to_center_km"]
X_raw = df[features].to_numpy(float)
y = df["price_billion_vnd"].to_numpy()
Xs = (X_raw - X_raw.mean(0)) / X_raw.std(0)
X = np.hstack([np.ones((len(Xs), 1)), Xs])
m = len(y)


def run_gd(alpha, iters=60):
    theta, hist = np.zeros(X.shape[1]), []
    for _ in range(iters):
        theta = theta - alpha / m * X.T @ (X @ theta - y)
        hist.append((X @ theta - y) @ (X @ theta - y) / (2 * m))
    return np.array(hist)


fig, ax = plt.subplots(figsize=(8, 4.5))
for alpha in [0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 1.2]:
    hist = run_gd(alpha)
    ax.plot(np.clip(hist, None, 1e3), label=f"α = {alpha}")
ax.set_yscale("log")
ax.set_xlabel("vòng lặp")
ax.set_ylabel("J(θ)")
ax.legend(ncol=2)
ax.set_title("Quét learning rate: α quá lớn → J tăng vọt")
plt.show()

### Một vài lời khuyên

- **Chuẩn hoá feature** trước khi chạy GD (về $[-1, 1]$, $[0, 1]$ hoặc z-score) — hội tụ nhanh hơn nhiều.
- Bắt đầu với learning rate nhỏ, tăng dần (×3) đến khi $J$ bắt đầu dao động/tăng, rồi lùi lại.
- **Luôn vẽ $J$ theo vòng lặp.** $J$ phải giảm sau **mỗi** vòng lặp (với batch GD). Nếu $J$ tăng → giảm $\alpha$ (hoặc có bug trong gradient — kiểm tra bằng gradient check).
- Dữ liệu lớn → dùng **mini-batch GD** (bài 04).
- Dùng **momentum** / Adam để tăng tốc (chương 02 bài 05, chương 28).

## 2. Normal Equation

Normal Equation là phương pháp tìm nghiệm của Linear Regression **không cần vòng lặp**, **không cần chọn learning rate**, và **không bắt buộc scaling dữ liệu**:

$$\theta = (X^\top X)^{-1}X^\top y$$

### Chứng minh *(bổ sung)*

$J(\theta) = \frac{1}{2m}(X\theta - y)^\top(X\theta - y)$ là hàm lồi bậc hai. Tại cực tiểu, gradient bằng 0 (chương 02, bài 04):

$$\nabla J(\theta) = \frac{1}{m}X^\top(X\theta - y) = 0 \;\Longleftrightarrow\; X^\top X\,\theta = X^\top y \;\Longleftrightarrow\; \theta = (X^\top X)^{-1}X^\top y$$

Phương trình $X^\top X\theta = X^\top y$ được gọi là **phương trình chuẩn (normal equation)**. Tham khảo thêm: [Derivation of the Normal Equation](https://eli.thegreenplace.net/2014/derivation-of-the-normal-equation-for-linear-regression).

### Ý nghĩa hình học *(bổ sung)*

$X\theta$ là một tổ hợp tuyến tính các **cột** của $X$ — tức một điểm trong **không gian cột** của $X$. Normal equation tìm điểm $\hat y = X\theta$ **gần $y$ nhất** — chính là **hình chiếu vuông góc** của $y$ lên không gian cột. Vì vậy vector residual $y - \hat y$ **vuông góc** với mọi cột của $X$: $X^\top(y - X\theta) = 0$.

In [ ]:
theta_ne = np.linalg.solve(X.T @ X, X.T @ y)          # giải hệ thay vì nghịch đảo (chương 02, bài 03)
residual = y - X @ theta_ne
print("θ (normal equation):", theta_ne.round(4))
print("Xᵀ · residual ≈ 0 :", (X.T @ residual).round(10))

theta_gd = np.zeros(X.shape[1])
for _ in range(2000):
    theta_gd -= 0.3 / m * X.T @ (X @ theta_gd - y)
print("θ (GD 2000 vòng)   :", theta_gd.round(4))

> 💡 Trong code, **không** tính `np.linalg.inv(X.T @ X) @ X.T @ y`. Dùng `np.linalg.solve` (giải hệ) hoặc `np.linalg.lstsq` (ổn định nhất, xử lý cả trường hợp suy biến) — đây cũng là cách `sklearn.linear_model.LinearRegression` làm.

## 3. So sánh Gradient Descent và Normal Equation

| | Gradient Descent | Normal Equation |
|---|---|---|
| Learning rate | cần chọn $\alpha$ | không cần |
| Vòng lặp | cần nhiều vòng lặp | không (một lần giải) |
| Scaling feature | rất nên làm | không bắt buộc |
| Chi phí tính toán | $O(k\,m\,n)$ — $k$ vòng lặp, mỗi vòng một phép nhân $X^\top(X\theta - y)$ | $O(m\,n^2 + n^3)$ — tạo $X^\top X$ rồi giải hệ $n \times n$ |
| Khi **số feature $n$** rất lớn (> ~10 000) | ✅ vẫn tốt | ❌ chậm ($n^3$), tốn bộ nhớ ($n^2$) |
| Khi **số mẫu $m$** rất lớn, không vừa RAM | ✅ mini-batch GD | ❌ cần cả $X$ (hoặc tích luỹ $X^\top X$) |
| Áp dụng cho model khác | ✅ mọi model khả vi (logistic, neural net...) | ❌ chỉ linear regression (và ridge) |

> 📝 *Đính chính so với bản gốc:* chi phí $O(n^3)$ của normal equation phụ thuộc vào **số feature** $n$, không phải số dữ liệu. Với hàng triệu mẫu nhưng chỉ vài chục feature, normal equation vẫn rất nhanh; nó chỉ chậm khi số **feature** lên tới hàng chục nghìn.

In [ ]:
rng = np.random.default_rng(0)
rows = []
for n_feat in [10, 100, 1000, 3000]:
    A = rng.normal(size=(5000, n_feat))
    b = A @ rng.normal(size=n_feat) + rng.normal(size=5000)
    start = time.perf_counter()
    np.linalg.solve(A.T @ A, A.T @ b)
    rows.append((n_feat, (time.perf_counter() - start) * 1000))
pd.DataFrame(rows, columns=["số feature n (m = 5000)", "normal equation (ms)"])

## 4. Khi $X^\top X$ không khả nghịch *(bổ sung)*

$X^\top X$ suy biến (không khả nghịch) khi:
1. **Feature dư thừa tuyến tính** (multicollinearity hoàn hảo): vd có cả `diện tích (m²)` và `diện tích (feet²)`; hay one-hot đủ $K$ cột + cột bias (dummy trap — chương 04).
2. **Nhiều feature hơn số mẫu** ($n > m$).

Cách xử lý: bỏ feature dư thừa; dùng **pseudo-inverse** / `lstsq` (cho nghiệm có chuẩn nhỏ nhất); hoặc **regularization** — Ridge giải $(X^\top X + \lambda I)\theta = X^\top y$, luôn khả nghịch với $\lambda > 0$ (bài 07).

In [ ]:
X_dup = np.hstack([X, X[:, [1]] * 10.764])            # thêm cột diện tích theo feet² — trùng thông tin
print("rank:", np.linalg.matrix_rank(X_dup), "/", X_dup.shape[1], "| cond(XᵀX) =", f"{np.linalg.cond(X_dup.T @ X_dup):.2e}")
theta_lstsq, *_ = np.linalg.lstsq(X_dup, y, rcond=None)
print("lstsq vẫn cho dự đoán đúng:", np.allclose(X_dup @ theta_lstsq, X @ theta_ne))
print("nhưng hệ số của 2 cột diện tích bị 'chia nhau':", theta_lstsq[[1, -1]].round(4))

## 🧠 Kiểm tra nhanh

1. Có 2 triệu mẫu và 20 feature. Normal equation có chậm không?
2. Có 5 000 mẫu và 50 000 feature (vd: văn bản). Nên dùng phương pháp nào? $X^\top X$ có khả nghịch không?
3. Giải thích vì sao residual vuông góc với mọi cột của $X$ tại nghiệm tối ưu.
4. Cost $J$ tăng sau một vòng lặp batch GD. Hai nguyên nhân có thể là gì?

## Tài liệu tham khảo

[1] [CS229 — Machine Learning Course](http://cs229.stanford.edu) · [2] Gilbert Strang — *Introduction to Linear Algebra*, chương 4 (projections & least squares)

---
<sub>Bài gốc: *Gradient Descent Tips và Normal Equation* — [AIVN-Machine-Learning](https://github.com/thanhhff/AIVN-Machine-Learning), Tuần 3 (MIT License). Đã đính chính độ phức tạp (theo số feature), bổ sung: thực nghiệm quét learning rate, chứng minh, ý nghĩa hình học, đo thời gian, trường hợp suy biến.</sub>